# OFFSET / FETCH NEXT — paginacja w T-SQL, notatki referencyjne

Przykłady na modelu: `fact_Sprzedaz` (ID_Klienta, DataSprzedazy, Kwota), `dim_Klienci` (ID_Klienta, Nazwa).

## 1. Składnia podstawowa

```sql
SELECT <kolumny>
FROM <tabela>
ORDER BY <kolumna(y)>   -- WYMAGANE, bez wyjątku
OFFSET <liczba_wierszy_do_pominięcia> ROWS
FETCH NEXT <liczba_wierszy_do_zwrócenia> ROWS ONLY;
```

**`ORDER BY` jest obowiązkowy** — `OFFSET`/`FETCH` bez sensownego sortowania nie ma znaczenia ("pomiń pierwsze N wierszy" zakłada jakąś kolejność; bez `ORDER BY` SQL Server zwróci błąd składniowy, nie zignoruje po cichu).

### Przykład — druga strona wyników, po 20 wierszy na stronę

```sql
SELECT k.ID_Klienta, k.Nazwa, s.DataSprzedazy, s.Kwota
FROM fact_Sprzedaz s
JOIN dim_Klienci k ON k.ID_Klienta = s.ID_Klienta
ORDER BY s.DataSprzedazy DESC
OFFSET 20 ROWS
FETCH NEXT 20 ROWS ONLY;
```

Pomija pierwsze 20 wierszy (strona 1), zwraca kolejne 20 (strona 2).

### Parametryzacja pod dynamiczną paginację

```sql
DECLARE @NumerStrony INT = 3;
DECLARE @RozmiarStrony INT = 20;

SELECT k.ID_Klienta, k.Nazwa, s.DataSprzedazy, s.Kwota
FROM fact_Sprzedaz s
JOIN dim_Klienci k ON k.ID_Klienta = s.ID_Klienta
ORDER BY s.DataSprzedazy DESC
OFFSET (@NumerStrony - 1) * @RozmiarStrony ROWS
FETCH NEXT @RozmiarStrony ROWS ONLY;
```

Standardowy wzorzec do przekazania z aplikacji/Pythona jako parametry `sp_executesql` (patrz notatnik o dynamicznym SQL) — bezpieczne, bo `@NumerStrony`/`@RozmiarStrony` to prawdziwe wartości danych, nie struktura zapytania.

## 2. `OFFSET` bez `FETCH` — sam pomija, bez limitu

```sql
SELECT * FROM fact_Sprzedaz
ORDER BY DataSprzedazy
OFFSET 100 ROWS;   -- bez FETCH NEXT — zwraca WSZYSTKO od 101. wiersza do końca
```

`FETCH NEXT` jest **opcjonalny** — sam `OFFSET` pomija podaną liczbę wierszy i zwraca resztę bez ograniczenia. Rzadziej praktycznie użyteczne (zwykle chcesz obu razem przy paginacji), ale warto wiedzieć, że to poprawna, samodzielna składnia.

**`FETCH FIRST` jako synonim `FETCH NEXT`** — SQL Server akceptuje oba słowa jako identyczne (`FETCH FIRST 20 ROWS ONLY` = `FETCH NEXT 20 ROWS ONLY`) — czysto stylistyczna różnica, zwyczajowo `FIRST` bywa używane przy `OFFSET 0`, `NEXT` przy paginacji kolejnych stron, ale silnik nie rozróżnia ich funkcjonalnie.

## 3. Krytyczny problem wydajnościowy — głęboka paginacja jest kosztowna

To jest najważniejsza rzecz do zrozumienia przed użyciem `OFFSET`/`FETCH` w produkcyjnym kodzie na dużych tabelach.

**`OFFSET N ROWS` nie "przeskakuje" do N-tego wiersza za darmo — silnik musi fizycznie posortować i policzyć pierwsze N wierszy, żeby wiedzieć, gdzie zacząć zwracać wynik.** Im głębiej w paginacji (im większe `N`), tym więcej pracy silnik wykonuje, mimo że finalnie zwraca tylko np. 20 wierszy.

**Konkretna konsekwencja:** strona 1 (`OFFSET 0`) jest tania. Strona 5000 (`OFFSET 100000`) na dużej tabeli faktów może być **drastycznie wolniejsza**, mimo że zwraca tę samą liczbę wierszy co strona 1 — silnik i tak musi przetworzyć/policzyć 100 020 wierszy, żeby dostarczyć ostatnie 20. To jest fundamentalnie inny profil kosztu niż mogłoby się wydawać z samej składni "pomiń i pobierz".

### Alternatywa dla głębokiej paginacji — keyset pagination (seek method)

Zamiast liczyć "pomiń N wierszy", zapamiętaj **wartość ostatniego wiersza poprzedniej strony** i filtruj względem niej bezpośrednio — to pozwala silnikowi użyć indeksu do bezpośredniego "przeskoczenia" (index seek), zamiast liczenia wierszy od początku:

```sql
-- Strona 1
SELECT TOP 20 s.ID_Sprzedazy, s.DataSprzedazy, s.Kwota
FROM fact_Sprzedaz s
ORDER BY s.DataSprzedazy DESC, s.ID_Sprzedazy DESC;

-- Zapamiętaj DataSprzedazy i ID_Sprzedazy ostatniego wiersza strony 1, np. '2025-03-15', 48291

-- Strona 2 — filtruj względem zapamiętanej pozycji, NIE licz OFFSET
SELECT TOP 20 s.ID_Sprzedazy, s.DataSprzedazy, s.Kwota
FROM fact_Sprzedaz s
WHERE (s.DataSprzedazy < '2025-03-15')
   OR (s.DataSprzedazy = '2025-03-15' AND s.ID_Sprzedazy < 48291)
ORDER BY s.DataSprzedazy DESC, s.ID_Sprzedazy DESC;
```

**Dlaczego to jest szybsze:** warunek `WHERE` na kolumnie objętej indeksem pozwala silnikowi bezpośrednio "wskoczyć" we właściwe miejsce posortowanych danych (index seek) — koszt jest praktycznie **stały, niezależnie od tego, jak głęboko jesteś w paginacji**, w przeciwieństwie do `OFFSET`, gdzie koszt rośnie liniowo z głębokością.

**Kompromis:** keyset pagination nie pozwala łatwo "skoczyć na stronę 47" (bo nie liczysz numerów stron, tylko poruszasz się względem konkretnej pozycji) — nadaje się do "poprzednia/następna strona" (jak scroll w interfejsie), nie do numerowanej nawigacji stron. Jeśli użytkownik faktycznie potrzebuje kliknąć "strona 47" z listy numerów stron, `OFFSET`/`FETCH` jest prostsze do zaimplementowania, kosztem wydajności przy głębokich stronach.

**Powiązanie z DAX:** to jest dokładnie ten sam problem koncepcyjnie, co funkcja `ISONORAFTER` w DAX (emulacja "START AT" dla paginacji przez porównanie wielokolumnowe) — różne silniki, ten sam fundamentalny kompromis między prostotą numerowanych stron a wydajnością głębokiej nawigacji.

## 4. Starsza alternatywa — paginacja przez `ROW_NUMBER()` (SQL Server 2008-2011, przed `OFFSET`/`FETCH`)

Warto rozpoznawać ten wzorzec w starszym kodzie — funkcjonalnie równoważny `OFFSET`/`FETCH`, ale zapisany przez CTE i `ROW_NUMBER()` (funkcję okienkową z poprzedniego notatnika):

```sql
WITH Ponumerowane AS (
    SELECT
        s.ID_Sprzedazy, s.DataSprzedazy, s.Kwota,
        ROW_NUMBER() OVER (ORDER BY s.DataSprzedazy DESC) AS Numer
    FROM fact_Sprzedaz s
)
SELECT ID_Sprzedazy, DataSprzedazy, Kwota
FROM Ponumerowane
WHERE Numer BETWEEN 21 AND 40;   -- odpowiednik OFFSET 20 ROWS FETCH NEXT 20 ROWS ONLY
```

**`OFFSET`/`FETCH` (SQL Server 2012+) to czytelniejszy, krótszy zapis dokładnie tego samego** — nie ma między nimi istotnej różnicy wydajnościowej (oba mają ten sam fundamentalny problem głębokiej paginacji z sekcji 3, bo oba wymagają posortowania/ponumerowania wszystkich wierszy do punktu odcięcia). Używaj `OFFSET`/`FETCH` w nowym kodzie — prostsze, standardowe, ta sama wydajność. Rozpoznawaj wzorzec `ROW_NUMBER`+`WHERE BETWEEN` w starszym kodzie jako to samo zadanie zapisane starszą składnią, nie jako coś fundamentalnie innego.

## 5. Podsumowanie — kiedy co

| Sytuacja | Rozwiązanie |
|---|---|
| Standardowa paginacja z numerami stron, umiarkowana głębokość | `OFFSET ... ROWS FETCH NEXT ... ROWS ONLY` |
| Bardzo głęboka paginacja na dużej tabeli (dziesiątki/setki tysięcy wierszy w OFFSET) | Keyset pagination (filtr `WHERE` względem ostatniej pozycji) zamiast `OFFSET` |
| Nawigacja tylko "poprzednia/następna" (scroll, infinite scroll) | Keyset pagination — naturalne dopasowanie do tego wzorca UI |
| Nawigacja z klikalnymi numerami stron (1, 2, 3, ... 47) | `OFFSET`/`FETCH` — prostsze do zaimplementowania, akceptuj koszt przy głębokich stronach |
| Starszy kod z `ROW_NUMBER()` + `WHERE BETWEEN` | Rozpoznaj jako odpowiednik `OFFSET`/`FETCH` — ten sam koszt, starsza składnia |
| Chcesz po prostu pominąć N wierszy bez limitu | `OFFSET N ROWS` bez `FETCH NEXT` |